In [ ]:
import os
import random
from datetime import datetime, timedelta
import pandas as pd
from faker import Faker
from google.cloud import bigquery
from google.oauth2 import service_account

project_id = "mi-proyecto-net-510916"
dataset_id = "tu_empresa"
credentials_path = "../../credentials/service-account.json"

credentials = service_account.Credentials.from_service_account_file(credentials_path)

client = bigquery.Client(credentials=credentials, project=project_id)
fake = Faker('es_ES')

table_ref_base = f"{project_id}.{dataset_id}"
now = datetime.now()

print("Generando datos sintéticos...")

categories_data = [
    {"category_id": "CAT_01", "name": "Smartphones", "description": "Teléfonos inteligentes y móviles de última generación"},
    {"category_id": "CAT_02", "name": "Laptops", "description": "Ordenadores portátiles y ultrabooks"},
    {"category_id": "CAT_03", "name": "Periféricos", "description": "Teclados, ratones, monitores y accesorios"},
    {"category_id": "CAT_04", "name": "Wearables", "description": "Relojes inteligentes y pulseras de actividad"},
    {"category_id": "CAT_05", "name": "Audio", "description": "Auriculares inalámbricos y altavoces bluetooth"}
]
df_categories = pd.DataFrame(categories_data)

products_data = []
cat_ids = [c["category_id"] for c in categories_data]
for i in range(1, 71):
    prod_id = f"PROD_{i:03d}"
    cost = round(random.uniform(20.0, 800.0), 2)
    price = round(cost * random.uniform(1.3, 1.8), 2)
    products_data.append({
        "product_id": prod_id,
        "name": f"Tech Device {i}",
        "category_id": random.choice(cat_ids),
        "price": price,
        "cost": cost,
        "stock": random.randint(10, 500),
        "is_active": True
    })
df_products = pd.DataFrame(products_data)

countries = ["Spain", "France", "Germany", "Italy", "Portugal"]
channels = ["organic", "paid ads", "social media", "email", "referral"]
customers_data = []
for i in range(1, 501):
    cust_id = f"CUST_{i:04d}"
    created_date = now - timedelta(days=random.randint(365, 730))
    customers_data.append({
        "customer_id": cust_id,
        "first_name": fake.first_name(),
        "last_name": fake.last_name(),
        "email": fake.email(),
        "country": random.choice(countries),
        "city": fake.city(),
        "acquisition_channel": random.choice(channels),
        "created_at": created_date.isoformat()
    })
df_customers = pd.DataFrame(customers_data)

orders_data = []
order_items_data = []
payments_data = []
reviews_data = []

statuses = ["pending", "confirmed", "shipped", "delivered", "cancelled", "returned"]
payment_methods = ["credit_card", "paypal", "apple_pay", "klarna"]

item_counter = 1
payment_counter = 1
review_counter = 1

for i in range(1, 2001):
    order_id = f"ORD_{i:05d}"
    cust = random.choice(customers_data)
    status = random.choices(statuses, weights=[10, 20, 30, 30, 5, 5], k=1)[0]
    
    order_date = now - timedelta(days=random.randint(1, 365))
    shipped_date = order_date + timedelta(days=random.randint(1, 3)) if status in ["shipped", "delivered", "returned"] else None
    delivered_date = shipped_date + timedelta(days=random.randint(2, 5)) if status == "delivered" else None
    
    orders_data.append({
        "order_id": order_id,
        "customer_id": cust["customer_id"],
        "status": status,
        "shipping_address": fake.address().replace("\n", ", "),
        "shipping_country": cust["country"],
        "order_date": order_date.isoformat(),
        "shipped_date": shipped_date.isoformat() if shipped_date else None,
        "delivered_date": delivered_date.isoformat() if delivered_date else None
    })
    
    num_items = random.choices([1, 2, 3, 4], weights=[20, 40, 30, 10], k=1)[0]
    selected_products = random.sample(products_data, num_items)
    
    order_total = 0
    for prod in selected_products:
        qty = random.randint(1, 3)
        unit_price = prod["price"]
        discount = round(random.choice([0.0, 0.0, 0.0, 0.05, 0.10]), 2)
        
        order_items_data.append({
            "order_item_id": f"ITEM_{item_counter:06d}",
            "order_id": order_id,
            "product_id": prod["product_id"],
            "quantity": qty,
            "unit_price": unit_price,
            "discount": discount
        })
        
        subtotal = (unit_price * qty) * (1 - discount)
        order_total += subtotal
        
        if status == "delivered" and random.random() < 0.35:
            reviews_data.append({
                "review_id": f"REV_{review_counter:06d}",
                "order_item_id": f"ITEM_{item_counter:06d}",
                "customer_id": cust["customer_id"],
                "rating": random.choices([3, 4, 5], weights=[10, 40, 50], k=1)[0],
                "comment": fake.sentence(),
                "created_at": (delivered_date + timedelta(days=random.randint(1, 10))).isoformat()
            })
            review_counter += 1
            
        item_counter += 1

    pay_status = "completed" if status in ["confirmed", "shipped", "delivered"] else ("refunded" if status in ["cancelled", "returned"] else "pending")
    payments_data.append({
        "payment_id": f"PAY_{payment_counter:06d}",
        "order_id": order_id,
        "payment_method": random.choice(payment_methods),
        "status": pay_status,
        "amount": round(order_total, 2),
        "payment_date": order_date.isoformat()
    })
    payment_counter += 1

df_orders = pd.DataFrame(orders_data)
df_order_items = pd.DataFrame(order_items_data)
df_payments = pd.DataFrame(payments_data)
df_reviews = pd.DataFrame(reviews_data)

print("Datos sintéticos generados con éxito. Subiendo a BigQuery...")

def load_df_to_bq(df, table_name):
    table_ref = f"{table_ref_base}.{table_name}"
    job_config = bigquery.LoadJobConfig(write_disposition="WRITE_APPEND")
    job = client.load_table_from_dataframe(df, table_ref, job_config=job_config)
    job.result()
    print(f"-> Tabla {table_name} poblada correctamente ({len(df)} filas cargadas).")

load_df_to_bq(df_categories, "categories")
load_df_to_bq(df_products, "products")
load_df_to_bq(df_customers, "customers")
load_df_to_bq(df_orders, "orders")
load_df_to_bq(df_order_items, "order_items")
load_df_to_bq(df_payments, "payments")
if not df_reviews.empty:
    load_df_to_bq(df_reviews, "reviews")

print("¡Proceso de generación y carga completado al 100%!")

Generando datos sintéticos...
Datos sintéticos generados con éxito. Subiendo a BigQuery...


c:\Users\y0rd4\trabajo_SQL\venv\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


-> Tabla categories poblada correctamente (5 filas cargadas).


c:\Users\y0rd4\trabajo_SQL\venv\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


-> Tabla products poblada correctamente (70 filas cargadas).


...

-> Tabla order_items poblada correctamente (4617 filas cargadas).


c:\Users\y0rd4\trabajo_SQL\venv\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


-> Tabla payments poblada correctamente (2000 filas cargadas).


c:\Users\y0rd4\trabajo_SQL\venv\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


-> Tabla reviews poblada correctamente (462 filas cargadas).
¡Proceso de generación y carga completado al 100%!
